# Chuẩn đoán bệnh tiểu đường

Trong phần trước, chúng ta đã tiến hành các bước sau:
1. Định nghĩa vấn đề
    + Mô tả vấn đề
    + Xác định đầu vào, đầu ra, loại bài toán
2. Chuẩn bị vấn đề:
    + Tải thư viện và dữ liệu
3. Phân tích dữ liệu:
    + Hiển thị thông tin dữ liệu
    + Thông tin thống kê trên các thuộc tính dữ liệu
    + Mối tương quan giữa các thuộc tính
5. Chia dữ liệu
6. Chuẩn bị dữ liệu:
    + Làm sạch dữ liệu (tạo bảng dữ liệu chỉ có thuộc tính nhập, xuất, xử lý dữ liệu thiếu và trùng lặp)
    + Biến đổi dữ liệu (chuẩn hóa)


**Kết quả:**
+ exps/data:
  + train.xlsx
  + test.xls
+ feature1:
  
  + scale_columns.npz (chứa cột biến đổi)

  + Feature MinMax:
    + minmax_scaler.joblib
    + feat_minmax.npz
    + df_minmax.xlsx

+ Feature Standard:
    + standard_scaler.joblib
    + feat_standard.npz
    + df_standard.xlsx

## Khởi tạo thí nghiệm 

### Khai báo thư viện

In [1]:
import os, sys
from IPython import display
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import ticker

import pandas as pd
import seaborn as sns
import joblib
import pprint
import random

from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
import sklearn

from sklearn.metrics import accuracy_score , ConfusionMatrixDisplay, confusion_matrix

import warnings
%matplotlib inline


warnings.filterwarnings("ignore")

c:\Python313\Lib\site-packages\xgboost\compat.py:105: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### Tham số thực nghiệm

In [2]:
params = {}

params['exps_dir'] = '../exps'
params['exp_name'] = 'pima_minmax'

params['exp_root'] = f'{params["exps_dir"]}/result1_minmax'
params["save_dir"]  = f'{params["exps_dir"]}/result1_{params["exp_name"]}'

params["data_path"]  = f'{params["exps_dir"]}/feature1/df_minmax.xlsx'

params['k_fold'] = 5
params['random_state'] = 42

print("params: ")
for k in params: print(f'+ {k}: {params[k]}')

random.seed(params['random_state'])
os.environ['PYTHONHASHSEED'] = str(params['random_state'])
np.random.seed(params['random_state'])

params: 
+ exps_dir: ../exps
+ exp_name: pima_minmax
+ exp_root: ../exps/result1_minmax
+ save_dir: ../exps/result1_pima_minmax
+ data_path: ../exps/feature1/df_minmax.xlsx
+ k_fold: 5
+ random_state: 42


## 5. Dữ liệu kiểm nghiệm
Chuẩn bị dữ liệu kiểm nghiệm theo phương pháp hold-out:
- Chia tập dữ liệu thành 2 phần train/test với tỉ lệ 7/3
- Tập train sẽ được dùng để huấn luyện mô hình và điều chỉnh tham số, với hai cách: 
    - Hold-out (tiếp tục chia 7/3 với train/valid)
    - k-fold (chia thành k phần đều nhau với k-1 phần cho train/1 phần cho valid)
    - Trong đó, train là dùng huấn luyện và valid để tối ưu tham số
- Tập test dùng để kiểm nghiệm lại độ hiệu quả của thuật toán sau khi chọn mô hình tối ưu

In [3]:
# tải dữ liệu
df = pd.read_excel(params['data_path'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 537 entries, 0 to 536
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               537 non-null    float64
 1   Glucose                   537 non-null    float64
 2   BloodPressure             537 non-null    float64
 3   SkinThickness             537 non-null    float64
 4   Insulin                   537 non-null    float64
 5   BMI                       537 non-null    float64
 6   DiabetesPedigreeFunction  537 non-null    float64
 7   Age                       537 non-null    float64
 8   Outcome                   537 non-null    int64  
dtypes: float64(8), int64(1)
memory usage: 37.9 KB


In [4]:
# Chia dữ liệu thành input/ouptu
x_train, y_train = df.values[:,:-1], df.values[:,-1].astype(int)
print(f'x_shape: {x_train.shape}, y_shape: {y_train.shape})')
print('Input: \n', x_train[:10,:])
print('Output: \n', y_train[:10])

x_shape: (537, 8), y_shape: (537,))
Input: 
 [[0.05882353 0.32903226 0.36734694 0.18181818 0.05288462 0.11656442
  0.07771136 0.01666667]
 [0.29411765 0.39354839 0.48979592 0.38181818 0.37379808 0.38241309
  0.03458582 0.11666667]
 [0.         0.58709677 0.44897959 0.61818182 0.28365385 0.49284254
  0.12254483 0.05      ]
 [0.23529412 0.56129032 0.44897959 0.23636364 0.18269231 0.30470348
  0.03501281 0.11666667]
 [0.05882353 0.38064516 0.06122449 0.54545455 0.08293269 0.51329243
  0.04483348 0.2       ]
 [0.11764706 0.24516129 0.28571429 0.25454545 0.12139423 0.21063395
  0.69214347 0.06666667]
 [0.17647059 0.54193548 0.55102041 0.38181818 0.13341346 0.0593047
  0.08112724 0.56666667]
 [0.05882353 0.50322581 0.40816327 0.43636364 0.17067308 0.34560327
  0.26216909 0.15      ]
 [0.         0.60645161 0.49225829 0.38181818 0.13341346 0.37014315
  0.36507259 0.06666667]
 [0.         0.52258065 0.44897959 0.38181818 0.13341346 0.13292434
  0.05465414 0.        ]]
Output: 
 [0 0 1 0 0 0 0 

## 6. Lượng giá thuật toán 

### 6.1 Baselines

In [ ]:
baseline_models = {
    'Logistic Regression': LogisticRegression(random_state=params["random_state"]),
    # 'Decision Tree': DecisionTreeClassifier(random_state=params["random_state"]),
    'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
    # 'K-Nearest Neighbors': KNeighborsClassifier(),
    'SVM': SVC(random_state=params["random_state"]),
    'XGBoost': XGBClassifier(random_state=params["random_state"]),
    'Random Forest': RandomForestClassifier(random_state=params["random_state"]),
}


kfold = KFold(n_splits=params['k_fold'], shuffle=True, random_state=params['random_state'])
for name, model in baseline_models.items():
    print (name)
    acc = cross_val_score(model, x_train, y_train, cv=kfold, scoring='accuracy').mean()
    f1 = cross_val_score(model, x_train, y_train, cv=kfold, scoring='f1').mean()
    auc = cross_val_score(model, x_train, y_train, cv=kfold, scoring='roc_auc').mean()
    print(f'+ Accuracy: {acc:.4f}\n F1-score: {f1:.4f}\n AUC: {auc:.4f}')
    print('-'*20)



Logistic Regression
+ Accuracy: 0.7654
 F1-score: 0.6032
 AUC: 0.8531
--------------------
Linear Discriminant Analysis
+ Accuracy: 0.7822
 F1-score: 0.6493
 AUC: 0.8535
--------------------
SVM
+ Accuracy: 0.7616
 F1-score: 0.6071
 AUC: 0.8504
--------------------
XGBoost
+ Accuracy: 0.7523
 F1-score: 0.6298
 AUC: nan
--------------------
Random Forest
+ Accuracy: 0.7654
 F1-score: 0.6419
 AUC: 0.8430
--------------------


### 6.2 Tinh chỉnh mô hình

In [6]:
tunning_results = {
    "best_clf"   : {},
    "best_score" : {},
}

tunning_models  = {}
tunning_params  = {}

# khởi tạo các tham số mặc định
tunning_models['Logistic Regression'] = LogisticRegression(random_state=params["random_state"])
tunning_params['Logistic Regression'] = {
    'C': [1, 10, 100],
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l1', 'l2']
}

tunning_models['Linear Discriminant Analysis'] = LinearDiscriminantAnalysis()
tunning_params['Linear Discriminant Analysis'] = {
    'solver': ['svd', 'lsqr', 'eigen'],
    'shrinkage': [None, 'auto']
}

tunning_models['Random Forest'] = RandomForestClassifier(random_state=params["random_state"])
tunning_params['Random Forest'] = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

for name, model in tunning_models.items():
    print(name)
    grid_clf = GridSearchCV(model, tunning_params[name], cv=kfold, scoring='accuracy')
    grid_result = grid_clf.fit(x_train, y_train)

    # store best model
    tunning_results["best_clf"][name] = grid_clf.best_estimator_

    # get search results
    tunning_results["best_score"][name] = grid_result.best_score_


    # information
    print(f'+ Best score: {grid_result.best_score_}')
    print(f'+ Best turnning params: {grid_result.best_params_}')
    print(f'+ Best full params: {grid_clf.best_estimator_.get_params()}')
    print()

Logistic Regression
+ Best score: 0.7858947732779509
+ Best turnning params: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}
+ Best full params: {'C': 10, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 100, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l1', 'random_state': 42, 'solver': 'liblinear', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}

Linear Discriminant Analysis
+ Best score: 0.7858601592246452
+ Best turnning params: {'shrinkage': 'auto', 'solver': 'lsqr'}
+ Best full params: {'covariance_estimator': None, 'n_components': None, 'priors': None, 'shrinkage': 'auto', 'solver': 'lsqr', 'store_covariance': False, 'tol': 0.0001}

Random Forest
+ Best score: 0.7784181377639321
+ Best turnning params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
+ Best full params: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sq

## 7. Kiểm nghiệm kết quả trên test

In [7]:
df_test = pd.read_csv(f'{params["exps_dir"]}/data/test.csv')
df_test

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,98,58,33,190,34.0,0.430,43,0
1,2,112,75,32,0,35.7,0.148,21,0
2,2,108,64,0,0,30.8,0.158,21,0
3,8,107,80,0,0,24.6,0.856,34,0
4,7,136,90,0,0,29.9,0.210,50,0
...,...,...,...,...,...,...,...,...,...
226,0,119,0,0,0,32.4,0.141,24,1
227,4,109,64,44,99,34.8,0.905,26,1
228,0,127,80,37,210,36.3,0.804,23,0
229,6,105,70,32,68,30.8,0.122,37,0


In [8]:
standard_scaler = joblib.load(f'{params["exps_dir"]}/feature1/minmax_scaler.joblib')
display.display(standard_scaler.__dict__)
scale_columns = dict(np.load(f'{params["exps_dir"]}/feature1/scale_columns.npz'))['columns']
scale_columns = scale_columns.tolist()
scale_columns

{'feature_range': (0, 1),
 'copy': True,
 'clip': False,
 'feature_names_in_': array(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age'], dtype=object),
 'n_features_in_': 8,
 'n_samples_seen_': 537,
 'scale_': array([0.05882353, 0.00645161, 0.01020408, 0.01818182, 0.00120192,
        0.0204499 , 0.42698548, 0.01666667]),
 'min_': array([ 0.        , -0.28387097, -0.24489796, -0.14545455, -0.01682692,
        -0.37218814, -0.03330487, -0.35      ]),
 'data_min_': array([ 0.   , 44.   , 24.   ,  8.   , 14.   , 18.2  ,  0.078, 21.   ]),
 'data_max_': array([ 17.  , 199.  , 122.  ,  63.  , 846.  ,  67.1 ,   2.42,  81.  ]),
 'data_range_': array([ 17.   , 155.   ,  98.   ,  55.   , 832.   ,  48.9  ,   2.342,
         60.   ])}

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age']

In [9]:
df_test[scale_columns] = standard_scaler.transform(df_test[scale_columns])
df_test.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,0.352941,0.348387,0.346939,0.454545,0.211538,0.323108,0.150299,0.366667,0
1,0.117647,0.438710,0.520408,0.436364,-0.016827,0.357873,0.029889,0.000000,0
2,0.117647,0.412903,0.408163,-0.145455,-0.016827,0.257669,0.034159,0.000000,0
3,0.470588,0.406452,0.571429,-0.145455,-0.016827,0.130879,0.332195,0.216667,0
4,0.411765,0.593548,0.673469,-0.145455,-0.016827,0.239264,0.056362,0.483333,0


In [10]:
x_test, y_test = df_test.values[:,:-1], df_test.values[:,-1].astype(int)

In [11]:
# baseline models
for name, model in baseline_models.items():
    model.fit(x_train, y_train)
    y_pred_test = model.predict(x_test)
    test_acc = accuracy_score(y_test, y_pred_test)

    print(name)
    print(f'+ Test accuracy: {test_acc:.4f}')

Logistic Regression
+ Test accuracy: 0.7489
Linear Discriminant Analysis
+ Test accuracy: 0.7403
SVM
+ Test accuracy: 0.7359
XGBoost
+ Test accuracy: 0.7273
Random Forest
+ Test accuracy: 0.7100


In [12]:
# Kiểm tra lại kết quả trên tập test (tunning models)
for name, model in tunning_results["best_clf"].items():
    model.fit(x_train, y_train)
    y_pred_test = model.predict(x_test)
    test_acc = accuracy_score(y_test, y_pred_test)

    print(name)
    print(f'+ Test accuracy: {test_acc:.4f}')

Logistic Regression
+ Test accuracy: 0.7446
Linear Discriminant Analysis
+ Test accuracy: 0.7532
Random Forest
+ Test accuracy: 0.7532


## 8. Lưu kết quả thí nghiệm

In [13]:
import os
save_dir = params["save_dir"]
os.makedirs(save_dir, exist_ok=True)

# Lưu notebook thành HTML (không sử dụng $cur_dir)
!jupyter nbconvert "model1.ipynb" --to html --output-dir "{save_dir}" --output "model1"

[NbConvertApp] Converting notebook model1.ipynb to html
[NbConvertApp] Writing 327569 bytes to ..\exps\result1_pima_minmax\model1.html
